<a href="https://colab.research.google.com/github/yourusername/visual-thesaurus-llm/blob/main/lora_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA Fine-Tuning for Language Models

This notebook demonstrates how to fine-tune a language model using LoRA (Low-Rank Adaptation), a parameter-efficient fine-tuning technique that significantly reduces the computational and memory requirements for fine-tuning large language models.

## What is LoRA?

LoRA freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture, greatly reducing the number of trainable parameters for downstream tasks.

### Key Benefits:
- **Memory Efficiency**: Requires significantly less GPU memory
- **Storage Efficiency**: LoRA adapters are typically 10-100MB vs. full models (GB)
- **Training Speed**: Faster training due to fewer parameters to update
- **Modularity**: Multiple LoRA adapters can be trained for different tasks

Let's get started!

## 1. Setup and Installation

First, let's install the necessary libraries:

In [ ]:
!pip install -q transformers==4.36.2 peft==0.7.1 datasets==2.15.0 accelerate==0.25.0 bitsandbytes==0.41.1 tqdm==4.66.1 torch==2.1.2 evaluate==0.4.1

Now, let's import the necessary libraries:

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from tqdm import tqdm

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Print GPU info if available
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load the Base Model

We'll use a small model for demonstration purposes. For a real application, you might want to use a larger model like GPT-J, LLaMA, or Mistral.

In [ ]:
# Model selection
model_name = "gpt2"  # A small model for demonstration
# For larger models, consider:
# model_name = "EleutherAI/gpt-neo-1.3B"
# model_name = "EleutherAI/gpt-j-6b"
# model_name = "meta-llama/Llama-2-7b-hf"  # Requires access
# model_name = "mistralai/Mistral-7B-v0.1"  # Requires access

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # Automatically determine device mapping
    torch_dtype=torch.float16,  # Use half precision
)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

## 3. Configure LoRA

Now, let's configure LoRA for our model. The key parameters are:

- **r**: Rank of the low-rank matrices (smaller = fewer parameters)
- **lora_alpha**: Scaling factor for the LoRA parameters
- **target_modules**: Which modules to apply LoRA to (depends on model architecture)
- **lora_dropout**: Dropout probability for LoRA layers

In [ ]:
# Define LoRA Config
lora_config = LoraConfig(
    r=8,                       # Rank of the update matrices
    lora_alpha=32,             # Alpha parameter for scaling
    target_modules=["c_attn"],  # For GPT-2, target the attention module
    lora_dropout=0.05,         # Dropout probability for LoRA layers
    bias="none",               # Add bias to LoRA layers
    task_type="CAUSAL_LM"      # Task type
)

# Apply LoRA to the model
peft_model = get_peft_model(model, lora_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

## 4. Prepare the Dataset

For this example, we'll use a small subset of the WikiText dataset. In a real application, you would use your own dataset.

In [ ]:
# Load a small dataset for demonstration
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")
print(f"Dataset loaded with {len(dataset)} examples")
print(f"Sample text: {dataset[0]['text'][:100]}...")

# Function to tokenize inputs
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Dataset tokenized with {len(tokenized_dataset)} examples")

# Prepare data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Not using masked language modeling
)

## 5. Train the Model

Now, let's set up the training arguments and train our model:

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./lora-model",
    learning_rate=3e-4,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,  # Effective batch size = 8
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,  # Use mixed precision training
    report_to="none",  # Disable wandb reporting
)

# Initialize Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Train the model
trainer.train()

## 6. Save the LoRA Adapter

After training, we can save the LoRA adapter (not the full model):

In [ ]:
# Save the LoRA adapter
peft_model.save_pretrained("./lora-adapter")
print("LoRA adapter saved to ./lora-adapter")

# Check the size of the saved adapter
!du -sh ./lora-adapter

## 7. Generate Text with the Fine-Tuned Model

Now, let's use our fine-tuned model to generate some text:

In [ ]:
# Define a function for text generation
def generate_text(prompt, max_length=100, temperature=0.7, do_sample=True):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate text
    with torch.no_grad():
        outputs = peft_model.generate(
            inputs.input_ids,
            max_length=max_length,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

# Test with some prompts
prompts = [
    "The history of artificial intelligence",
    "In recent years, machine learning has",
    "The future of natural language processing"
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    generated = generate_text(prompt)
    print(f"Generated: {generated}")
    print("-" * 50)

## 8. Load the LoRA Adapter Separately

You can also load the LoRA adapter separately, which is useful if you want to use it with a different instance of the base model:

In [ ]:
# Load the base model again
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load the LoRA adapter
loaded_peft_model = PeftModel.from_pretrained(
    base_model,
    "./lora-adapter",
    device_map="auto"
)

print("LoRA adapter loaded successfully")

# Test generation with the loaded model
prompt = "The most interesting aspect of deep learning is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = loaded_peft_model.generate(
        inputs.input_ids,
        max_length=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nPrompt: {prompt}")
print(f"Generated: {generated_text}")

## 9. Merge LoRA Weights with Base Model (Optional)

You can merge the LoRA weights with the base model to create a single model without the need for the adapter:

In [ ]:
# Merge weights
merged_model = loaded_peft_model.merge_and_unload()
print("LoRA weights merged with base model")

# Save the merged model
merged_model.save_pretrained("./merged-model")
tokenizer.save_pretrained("./merged-model")
print("Merged model saved to ./merged-model")

# Check the size of the merged model
!du -sh ./merged-model

## Conclusion

In this notebook, we've demonstrated how to:

1. Load a pre-trained language model
2. Configure LoRA for parameter-efficient fine-tuning
3. Prepare a dataset for fine-tuning
4. Train the model with LoRA
5. Save and load the LoRA adapter
6. Generate text with the fine-tuned model
7. Merge LoRA weights with the base model

LoRA is a powerful technique that allows you to fine-tune large language models with limited computational resources. It's particularly useful for adapting models to specific domains or tasks without the need for full fine-tuning.

For more advanced applications, consider:
- Using QLoRA (4-bit quantization + LoRA) for even more memory efficiency
- Training multiple LoRA adapters for different tasks
- Experimenting with different rank values and target modules

Happy fine-tuning!